In [9]:
from datetime import datetime, timedelta, timezone
import pandas as pd
import numpy as np
from cognite.client import CogniteClient
from cognite.client.data_classes import EventWrite
from cognite.client.exceptions import CogniteNotFoundError

# Initialize Cognite Client
client = CogniteClient()

# Local operational timezone: GMT-4
LOCAL_TZ = timezone(timedelta(hours=-4))

# D&I and Standum Process Constants
CANS_PER_SHORT_CAN = 5           # Cans lost per short can defect
CANS_PER_TRIMMER_JAM = 9        # Cans lost per trimmer jam
DOWNTIME_PER_SHORT_CAN_MIN = 5.0 # Minutes stopped per short can
DOWNTIME_PER_TRIM_JAM_MIN = 3.0 # Minutes stopped per trimmer jam
CAN_WEIGHT_KG = 0.0093           # Weight per aluminum can (~9.3g)
DATA_SET_ID = 5144187181631371

# Unified Machine Configurations
MACHINE_CONFIGS = [
    # --- PRINTERS ---
    {
        "code": "p11",
        "machine_type": "printer",
        "asset_ext_id": "SuperenvasesMQTT_L1_PRINTER",
        "nominal_capacity": 66000.0,
        "ts_prod": "PRINTER_L1_PROD_ACT_DISP",
        "ts_retract": "PRINTER_L1_RETRACT_ACT_DISP",
        "ts_blow_off": "PRINTER_L1_LAT_SOP_ACT_DISP",
    },
    {
        "code": "p31",
        "machine_type": "printer",
        "asset_ext_id": "SuperenvasesMQTT_L3_PRINTER_PRINTER31",
        "nominal_capacity": 66000.0,
        "ts_prod": "PRINTER_L31_PROD_ACT_DISP",
        "ts_retract": "PRINTER_L31_RETRACT_ACT_DISP",
        "ts_blow_off": "PRINTER_L31_LAT_SOP_ACT_DISP",
    },
    {
        "code": "p32",
        "machine_type": "printer",
        "asset_ext_id": "SuperenvasesMQTT_L3_PRINTER_PRINTER32",
        "nominal_capacity": 66000.0,
        "ts_prod": "PRINTER_L32_PROD_ACT_DISP",
        "ts_retract": "PRINTER_L32_RETRACT_ACT_DISP",
        "ts_blow_off": "PRINTER_L32_LAT_SOP_ACT_DISP",
    },
    # --- D&I MACHINERY ---
    {
        "code": "di11",
        "machine_type": "di",
        "asset_ext_id": "SuperenvasesMQTT_L1_DI_DI11",
        "ts_prod": "DI11_PROD_ACT_DISP",
        "ts_short_cans": "DI11_LATAS_CORTAS_ACT_DISP",
        "ts_trimmer_jams": "DI11_TRANC_TRIM_ACT_DISP",
    },
    {
        "code": "di12",
        "machine_type": "di",
        "asset_ext_id": "SuperenvasesMQTT_L1_DI_DI12",
        "ts_prod": "DI12_PROD_ACT_DISP",
        "ts_short_cans": "DI12_LATAS_CORTAS_ACT_DISP",
        "ts_trimmer_jams": "DI12_TRANC_TRIM_ACT_DISP",
    },
    {
        "code": "di14",
        "machine_type": "di",
        "asset_ext_id": "SuperenvasesMQTT_L1_DI_DI14",
        "ts_prod": "DI14_PROD_ACT_DISP",
        "ts_short_cans": "DI14_LATAS_CORTAS_ACT_DISP",
        "ts_trimmer_jams": "DI14_TRANC_TRIM_ACT_DISP",
    },
    {
        "code": "di15",
        "machine_type": "di",
        "asset_ext_id": "SuperenvasesMQTT_L1_DI_DI15",
        "ts_prod": "DI15_PROD_ACT_DISP",
        "ts_short_cans": "DI15_LATAS_CORTAS_ACT_DISP",
        "ts_trimmer_jams": "DI15_TRANC_TRIM_ACT_DISP",
    },
    {
        "code": "di17",
        "machine_type": "di",
        "asset_ext_id": "SuperenvasesMQTT_L1_DI_DI17",
        "ts_prod": "DI17_PROD_ACT_DISP",
        "ts_short_cans": "DI17_LATAS_CORTAS_ACT_DISP",
        "ts_trimmer_jams": "DI17_TRANC_TRIM_ACT_DISP",
    },
    {
        "code": "di18",
        "machine_type": "di",
        "asset_ext_id": "SuperenvasesMQTT_L1_DI_DI18",
        "ts_prod": "DI18_PROD_ACT_DISP",
        "ts_short_cans": "DI18_LATAS_CORTAS_ACT_DISP",
        "ts_trimmer_jams": "DI18_TRANC_TRIM_ACT_DISP",
    },
    # --- STANDUM MACHINERY ---
    {
        "code": "standum31",
        "machine_type": "standum",
        "asset_ext_id": "SuperenvasesMQTT_L3_STANDUM_STANDUM31",
        "ts_prod": "STANDUM31_PROD_ACT_DISP",
        "ts_short_cans": "STANDUM31_PROD_LATAS_CORTAS_ACT_DISP",
        "ts_trimmer_jams": "STANDUM31_TRANC_TRIM_ACUM_DISP",
    },
    {
        "code": "standum32",
        "machine_type": "standum",
        "asset_ext_id": "SuperenvasesMQTT_L3_STANDUM_STANDUM32",
        "ts_prod": "STANDUM32_PROD_ACT_DISP",
        "ts_short_cans": "STANDUM32_PROD_LATAS_CORTAS_ACT_DISP",
        "ts_trimmer_jams": "STANDUM32_TRANC_TRIM_ACUM_DISP",
    },
    {
        "code": "standum33",
        "machine_type": "standum",
        "asset_ext_id": "SuperenvasesMQTT_L3_STANDUM_STANDUM33",
        "ts_prod": "STANDUM33_PROD_ACT_DISP",
        "ts_short_cans": "STANDUM33_PROD_LATAS_CORTAS_ACT_DISP",
        "ts_trimmer_jams": "STANDUM33_TRANC_TRIM_ACUM_DISP",
    },
    {
        "code": "standum34",
        "machine_type": "standum",
        "asset_ext_id": "SuperenvasesMQTT_L3_STANDUM_STANDUM34",
        "ts_prod": "STANDUM34_PROD_ACT_DISP",
        "ts_short_cans": "STANDUM34_PROD_LATAS_CORTAS_ACT_DISP",
        "ts_trimmer_jams": "STANDUM34_TRANC_TRIM_ACUM_DISP",
    },
    {
        "code": "standum35",
        "machine_type": "standum",
        "asset_ext_id": "SuperenvasesMQTT_L3_STANDUM_STANDUM35",
        "ts_prod": "STANDUM35_PROD_ACT_DISP",
        "ts_short_cans": "STANDUM35_PROD_LATAS_CORTAS_ACT_DISP",
        "ts_trimmer_jams": "STANDUM35_TRANC_TRIM_ACUM_DISP",
    },
    {
        "code": "standum36",
        "machine_type": "standum",
        "asset_ext_id": "SuperenvasesMQTT_L3_STANDUM_STANDUM36",
        "ts_prod": "STANDUM36_PROD_ACT_DISP",
        "ts_short_cans": "STANDUM36_PROD_LATAS_CORTAS_ACT_DISP",
        "ts_trimmer_jams": "STANDUM36_TRANC_TRIM_ACUM_DISP",
    },
    {
        "code": "standum37",
        "machine_type": "standum",
        "asset_ext_id": "SuperenvasesMQTT_L3_STANDUM_STANDUM37",
        "ts_prod": "STANDUM37_PROD_ACT_DISP",
        "ts_short_cans": "STANDUM37_PROD_LATAS_CORTAS_ACT_DISP",
        "ts_trimmer_jams": "STANDUM37_TRANC_TRIM_ACUM_DISP",
    },
    {
        "code": "standum38",
        "machine_type": "standum",
        "asset_ext_id": "SuperenvasesMQTT_L3_STANDUM_STANDUM38",
        "ts_prod": "STANDUM38_PROD_ACT_DISP",
        "ts_short_cans": "STANDUM38_PROD_LATAS_CORTAS_ACT_DISP",
        "ts_trimmer_jams": "STANDUM38_TRANC_TRIM_ACUM_DISP",
    },
    # --- MINSTER PRESSES ---
    {
        "code": "minster_l1",
        "machine_type": "minster",
        "asset_ext_id": "SuperenvasesMQTT_L1_MINSTER",
        "nominal_capacity": 120000.0,
        "ts_golpes_bob": "MINSTER_L1_GOLPES_BOB_ACT_DISP",
        "ts_golpes_turno": "MINSTER_L1_GOLPES_TURNO_ACT_DISP",
    },
    {
        "code": "minster_l3",
        "machine_type": "minster",
        "asset_ext_id": "SuperenvasesMQTT_L3_MINSTER",
        "nominal_capacity": 120000.0,
        "ts_golpes_bob": "MINSTER_L3_GOLPES_BOB_ACT_DISP",
        "ts_golpes_turno": "MINSTER_L3_GOLPES_TURNO_ACT_DISP",
    },
    # --- ISPRAY LINE 1 (11 - 15) ---
    {"code": "ispray11", "machine_type": "ispray", "asset_ext_id": "SuperenvasesMQTT_L1_ISPRAY_ISPRAY11", "nominal_capacity": 30000.0, "ts_prod": "IPSPRAY_L1_11_PROD_ACT_DISP"},
    {"code": "ispray12", "machine_type": "ispray", "asset_ext_id": "SuperenvasesMQTT_L1_ISPRAY_ISPRAY12", "nominal_capacity": 30000.0, "ts_prod": "IPSPRAY_L1_12_PROD_ACT_DISP"},
    {"code": "ispray13", "machine_type": "ispray", "asset_ext_id": "SuperenvasesMQTT_L1_ISPRAY_ISPRAY13", "nominal_capacity": 30000.0, "ts_prod": "IPSPRAY_L1_13_PROD_ACT_DISP"},
    {"code": "ispray14", "machine_type": "ispray", "asset_ext_id": "SuperenvasesMQTT_L1_ISPRAY_ISPRAY14", "nominal_capacity": 30000.0, "ts_prod": "IPSPRAY_L1_14_PROD_ACT_DISP"},
    {"code": "ispray15", "machine_type": "ispray", "asset_ext_id": "SuperenvasesMQTT_L1_ISPRAY_ISPRAY15", "nominal_capacity": 30000.0, "ts_prod": "IPSPRAY_L1_15_PROD_ACT_DISP"},
    # --- ISPRAY LINE 3 (31 - 38) ---
    {"code": "ispray31", "machine_type": "ispray", "asset_ext_id": "SuperenvasesMQTT_L3_ISPRAY_ISPRAY31", "nominal_capacity": 30000.0, "ts_prod": "IPSPRAY_L3_31_PROD_ACT_DISP"},
    {"code": "ispray32", "machine_type": "ispray", "asset_ext_id": "SuperenvasesMQTT_L3_ISPRAY_ISPRAY32", "nominal_capacity": 30000.0, "ts_prod": "IPSPRAY_L3_32_PROD_ACT_DISP"},
    {"code": "ispray33", "machine_type": "ispray", "asset_ext_id": "SuperenvasesMQTT_L3_ISPRAY_ISPRAY33", "nominal_capacity": 30000.0, "ts_prod": "IPSPRAY_L3_33_PROD_ACT_DISP"},
    {"code": "ispray34", "machine_type": "ispray", "asset_ext_id": "SuperenvasesMQTT_L3_ISPRAY_ISPRAY34", "nominal_capacity": 30000.0, "ts_prod": "IPSPRAY_L3_34_PROD_ACT_DISP"},
    {"code": "ispray35", "machine_type": "ispray", "asset_ext_id": "SuperenvasesMQTT_L3_ISPRAY_ISPRAY35", "nominal_capacity": 30000.0, "ts_prod": "IPSPRAY_L3_35_PROD_ACT_DISP"},
    {"code": "ispray36", "machine_type": "ispray", "asset_ext_id": "SuperenvasesMQTT_L3_ISPRAY_ISPRAY36", "nominal_capacity": 30000.0, "ts_prod": "IPSPRAY_L3_36_PROD_ACT_DISP"},
    {"code": "ispray37", "machine_type": "ispray", "asset_ext_id": "SuperenvasesMQTT_L3_ISPRAY_ISPRAY37", "nominal_capacity": 30000.0, "ts_prod": "IPSPRAY_L3_37_PROD_ACT_DISP"},
    {"code": "ispray38", "machine_type": "ispray", "asset_ext_id": "SuperenvasesMQTT_L3_ISPRAY_ISPRAY38", "nominal_capacity": 30000.0, "ts_prod": "IPSPRAY_L3_38_PROD_ACT_DISP"},
]

HOUR_INTERVAL_MAP = {
    6: "6 a 7",    7: "7 a 8",    8: "8 a 9",    9: "9 a 10",
    10: "10 a 11", 11: "11 a 12", 12: "12 a 1",  13: "1 a 2",
    14: "2 a 3",   15: "3 a 4",   16: "4 a 5",   17: "5 a 6",
    18: "6 a 7",   19: "7 a 8",   20: "8 a 9",   21: "9 a 10",
    22: "10 a 11", 23: "11 a 12",  0: "12 a 1",   1: "1 a 2",
    2: "2 a 3",    3: "3 a 4",    4: "4 a 5",    5: "5 a 6",
}

In [10]:
def get_asset_id(client: CogniteClient, identifier: str) -> int:
    # 1. Try fetching directly by external_id
    try:
        asset = client.assets.retrieve(external_id=identifier)
        if asset:
            return asset.id
    except CogniteNotFoundError:
        pass  # Asset external_id not found; fallback to search

    # 2. Fallback search by name/query
    res = client.assets.search(query=identifier, limit=5)
    if res:
        return res[0].id

    raise ValueError(f"Could not find asset '{identifier}' in CDF.")


def calculate_hourly_counter_delta(client: CogniteClient, external_id: str, start_ms: int, end_ms: int):
    """
    Calculates step-by-step counter accumulation and handles mid-hour resets.
    """
    try:
        dps = client.time_series.data.retrieve(
            external_id=external_id,
            start=start_ms,
            end=end_ms,
            limit=None,
            ignore_unknown_ids=True,
        )

        if not dps or len(dps) == 0:
            print(f"  [Warning] [{external_id}] No datapoints found in window.")
            return 0.0, False

        # Helper for safely converting datapoint values to float
        def safe_float(val) -> float:
            try:
                return float(val) if val is not None else 0.0
            except (ValueError, TypeError):
                return 0.0

        first_value = safe_float(dps[0].value)
        last_value = safe_float(dps[-1].value)

        if len(dps) == 1:
            print(f"  [{external_id}] First: {first_value:,.1f} | Last: {last_value:,.1f} | Delta: 0.0 | Reset: False")
            return 0.0, False

        hourly_delta = 0.0
        reset_occurred = False

        for i in range(1, len(dps)):
            prev_val = safe_float(dps[i - 1].value)
            curr_val = safe_float(dps[i].value)
            step_diff = curr_val - prev_val

            if step_diff < 0:
                reset_occurred = True
                # Add current value upon reset assuming counter restarted near 0
                hourly_delta += max(0.0, curr_val)
            else:
                hourly_delta += step_diff

        print(f"  [{external_id}] First: {first_value:,.1f} | Last: {last_value:,.1f} | Delta: {hourly_delta:,.1f} | Reset: {reset_occurred}")
        return hourly_delta, reset_occurred

    except CogniteNotFoundError:
        print(f"  [Warning] TimeSeries '{external_id}' not found.")
        return 0.0, False
    except Exception as e:
        print(f"  [Error] Reading '{external_id}': {e}")
        return 0.0, False

In [11]:
def generate_printer_event(
    client: CogniteClient,
    cfg: dict,
    start_ms: int,
    end_ms: int,
    last_hour_start_local: datetime,
    asset_id: int,
):
    printer_code = cfg["code"]
    nominal_cap = cfg.get("nominal_capacity", 0.0)

    start_hour_local = last_hour_start_local.hour

    # 1. Determine shift and start hour (Day: 06:00-18:00 | Night: 18:00-06:00)
    shift_code = "day" if 6 <= start_hour_local < 18 else "night"
    shift_start_hour = 6 if shift_code == "day" else 18

    # 2. Adjust operational shift date for overnight hours (00:00 - 05:59)
    if start_hour_local < 6:
        shift_date = last_hour_start_local - timedelta(days=1)
    else:
        shift_date = last_hour_start_local

    date_str = shift_date.strftime("%Y%m%d")

    # 3. Compute entry slot index (01 to 12)
    hour_index = ((start_hour_local - shift_start_hour) % 24) + 1
    entry_slot = f"{hour_index:02d}"

    # 4. Get human-readable hour label from map
    hour_interval = HOUR_INTERVAL_MAP.get(
        start_hour_local,
        f"{start_hour_local} a {(start_hour_local + 1) % 24}",
    )

    # 5. Query CDF counters
    hourly_production, prod_reset = calculate_hourly_counter_delta(
        client, cfg["ts_prod"], start_ms, end_ms
    )
    hourly_retrac, retrac_reset = calculate_hourly_counter_delta(
        client, cfg["ts_retract"], start_ms, end_ms
    )
    blow_off, blowoff_reset = calculate_hourly_counter_delta(
        client, cfg["ts_blow_off"], start_ms, end_ms
    )

    # 6. Compute KPI metrics safely
    if nominal_cap > 0:
        downtime_minutes = max(
            0.0, round(60.0 - ((hourly_production * 60.0) / nominal_cap), 2)
        )
        efficiency = round((hourly_production * 100.0) / nominal_cap, 2)
    else:
        downtime_minutes = 60.0
        efficiency = 0.0

    event_ext_id = (
        f"report_{printer_code}_{date_str}_{shift_code}_entry_{entry_slot}"
    )

    # 7. Operational observations
    resets = []
    if prod_reset:
        resets.append("prod")
    if retrac_reset:
        resets.append("retrac")
    if blowoff_reset:
        resets.append("blow_off")

    obs_text = (
        f"Resets detected: {', '.join(resets)}"
        if resets
        else "Operación estándar"
    )

    # 8. Create event object
    report_event = EventWrite(
        external_id=event_ext_id,
        data_set_id=DATA_SET_ID,
        type="Production Report",
        subtype="Hourly Entry",
        start_time=start_ms,
        end_time=end_ms,
        description=f"Production Report {hour_interval} for Printer {printer_code.upper()}",
        asset_ids=[asset_id],
        metadata={
            "timezone": "GMT-4",
            "printer_code": printer_code,
            "shift": shift_code,
            "hour_interval": hour_interval,
            "hourly_production": str(hourly_production),
            "hourly_retrac": str(hourly_retrac),
            "blow_off": str(blow_off),
            "downtime_minutes": str(downtime_minutes),
            "efficiency": f"{efficiency:.2f}%",
            "observations": obs_text,
        },
    )

    # 9. Upsert to CDF
    res = client.events.upsert(report_event)
    ext_id = res.external_id if hasattr(res, "external_id") else res[0].external_id
    cdf_id = res.id if hasattr(res, "id") else res[0].id
    print(
        f"  --> Successfully posted Printer Event: '{ext_id}' (CDF ID: {cdf_id})"
    )


def generate_di_event(
    client: CogniteClient,
    cfg: dict,
    start_ms: int,
    end_ms: int,
    last_hour_start_local: datetime,
    asset_id: int,
):
    machine_code = cfg["code"]

    start_hour_local = last_hour_start_local.hour

    # 1. Determine shift and start hour (Day: 06:00-18:00 | Night: 18:00-06:00)
    shift_code = "day" if 6 <= start_hour_local < 18 else "night"
    shift_start_hour = 6 if shift_code == "day" else 18

    # 2. Adjust operational shift date for overnight hours (00:00 - 05:59)
    if start_hour_local < 6:
        shift_date = last_hour_start_local - timedelta(days=1)
    else:
        shift_date = last_hour_start_local

    date_str = shift_date.strftime("%Y%m%d")

    # 3. Compute entry slot index (01 to 12)
    hour_index = ((start_hour_local - shift_start_hour) % 24) + 1
    entry_slot = f"{hour_index:02d}"

    # 4. Get human-readable hour label from map
    hour_interval = HOUR_INTERVAL_MAP.get(
        start_hour_local,
        f"{start_hour_local} a {(start_hour_local + 1) % 24}",
    )

    # 5. Query CDF counters
    prod_count, prod_reset = calculate_hourly_counter_delta(
        client, cfg["ts_prod"], start_ms, end_ms
    )
    short_count, short_reset = calculate_hourly_counter_delta(
        client, cfg["ts_short_cans"], start_ms, end_ms
    )
    trim_count, trim_reset = calculate_hourly_counter_delta(
        client, cfg["ts_trimmer_jams"], start_ms, end_ms
    )

    # 6. Compute D&I Metrics
    cans_from_short = short_count * CANS_PER_SHORT_CAN
    cans_from_trim = trim_count * CANS_PER_TRIMMER_JAM

    downtime_short = short_count * DOWNTIME_PER_SHORT_CAN_MIN
    downtime_trim = trim_count * DOWNTIME_PER_TRIM_JAM_MIN
    total_downtime_min = min(60.0, downtime_short + downtime_trim)

    total_scrap_cans = cans_from_short + cans_from_trim
    merma_kg = round(total_scrap_cans * CAN_WEIGHT_KG, 2)

    total_produced_and_lost = prod_count + total_scrap_cans
    pct_merma = (
        round((total_scrap_cans / total_produced_and_lost * 100.0), 2)
        if total_produced_and_lost > 0
        else 0.0
    )
    pct_eficiencia = round(((60.0 - total_downtime_min) / 60.0 * 100.0), 2)

    # 7. Resets tracking
    resets = []
    if prod_reset:
        resets.append("prod")
    if short_reset:
        resets.append("short_cans")
    if trim_reset:
        resets.append("trimmer_jams")

    # 8. Operational Observations Text
    obs_parts = []
    if total_downtime_min == 0:
        obs_parts.append("Operación normal")
    elif short_count > 0 and trim_count > 0:
        obs_parts.append("Parada por latas cortas y trancamiento")
    elif short_count > 0:
        obs_parts.append("Parada por latas cortas")
    else:
        obs_parts.append("Parada por trancamiento trimmer")

    if resets:
        obs_parts.append(f"(Resets: {', '.join(resets)})")

    obs_text = " ".join(obs_parts)

    event_ext_id = (
        f"report_{machine_code}_{date_str}_{shift_code}_entry_{entry_slot}"
    )

    # 9. Create Event object
    report_event = EventWrite(
        external_id=event_ext_id,
        data_set_id=DATA_SET_ID,
        type="Production Report",
        subtype="Hourly Entry DI",
        start_time=start_ms,
        end_time=end_ms,
        description=f"Production Report {hour_interval} for D&I Machine {machine_code.upper()}",
        asset_ids=[asset_id],
        metadata={
            "timezone": "GMT-4",
            "machine_code": machine_code.upper(),
            "shift": shift_code,
            "hour_interval": hour_interval,
            "hourly_production": str(int(prod_count)),
            "short_cans_per_hour": str(int(short_count)),
            "trimmer_jams_per_hour": str(int(trim_count)),
            "cans_by_short_can": str(int(cans_from_short)),
            "cans_by_trimmer_jam": str(int(cans_from_trim)),
            "downtime_by_short_can_min": f"{downtime_short:.2f}",
            "downtime_by_trimmer_jam_min": f"{downtime_trim:.2f}",
            "total_downtime_min": f"{total_downtime_min:.2f}",
            "merma_kg": f"{merma_kg:.2f}",
            "pct_merma": f"{pct_merma:.2f}%",
            "pct_eficiencia": f"{pct_eficiencia:.2f}%",
            "observations": obs_text,
        },
    )

    # 10. Upsert event to CDF
    res = client.events.upsert(report_event)
    ext_id = res.external_id if hasattr(res, "external_id") else res[0].external_id
    cdf_id = res.id if hasattr(res, "id") else res[0].id
    print(
        f"  --> Successfully posted D&I Event: '{ext_id}' (CDF ID: {cdf_id})"
    )



def generate_standum_event(
    client: CogniteClient,
    cfg: dict,
    start_ms: int,
    end_ms: int,
    last_hour_start_local: datetime,
    asset_id: int,
):
    machine_code = cfg["code"]

    start_hour_local = last_hour_start_local.hour

    # 1. Determine shift and start hour (Day: 06:00-18:00 | Night: 18:00-06:00)
    shift_code = "day" if 6 <= start_hour_local < 18 else "night"
    shift_start_hour = 6 if shift_code == "day" else 18

    # 2. Adjust operational shift date for overnight hours (00:00 - 05:59)
    if start_hour_local < 6:
        shift_date = last_hour_start_local - timedelta(days=1)
    else:
        shift_date = last_hour_start_local

    date_str = shift_date.strftime("%Y%m%d")

    # 3. Compute entry slot index (01 to 12)
    hour_index = ((start_hour_local - shift_start_hour) % 24) + 1
    entry_slot = f"{hour_index:02d}"

    # 4. Get human-readable hour label from map
    hour_interval = HOUR_INTERVAL_MAP.get(
        start_hour_local,
        f"{start_hour_local} a {(start_hour_local + 1) % 24}",
    )

    # 5. Query CDF counters
    hourly_production, prod_reset = calculate_hourly_counter_delta(
        client, cfg["ts_prod"], start_ms, end_ms
    )
    short_count, short_reset = calculate_hourly_counter_delta(
        client, cfg["ts_short_cans"], start_ms, end_ms
    )
    trim_count, trim_reset = calculate_hourly_counter_delta(
        client, cfg["ts_trimmer_jams"], start_ms, end_ms
    )

    # 6. Compute Standum Metrics
    cans_from_short = short_count * CANS_PER_SHORT_CAN
    cans_from_trim = trim_count * CANS_PER_TRIMMER_JAM

    downtime_short = short_count * DOWNTIME_PER_SHORT_CAN_MIN
    downtime_trim = trim_count * DOWNTIME_PER_TRIM_JAM_MIN
    total_downtime_min = min(60.0, downtime_short + downtime_trim)

    total_scrap_cans = cans_from_short + cans_from_trim
    merma_kg = round(total_scrap_cans * CAN_WEIGHT_KG, 2)

    total_produced_and_lost = hourly_production + total_scrap_cans
    pct_merma = (
        round((total_scrap_cans / total_produced_and_lost * 100.0), 2)
        if total_produced_and_lost > 0
        else 0.0
    )
    pct_eficiencia = round(((60.0 - total_downtime_min) / 60.0 * 100.0), 2)

    # 7. Resets tracking
    resets = []
    if prod_reset:
        resets.append("prod")
    if short_reset:
        resets.append("short_cans")
    if trim_reset:
        resets.append("trimmer_jams")

    # 8. Operational Observations Text
    obs_parts = []
    if total_downtime_min == 0:
        obs_parts.append("Operación normal")
    elif short_count > 0 and trim_count > 0:
        obs_parts.append("Parada por latas cortas y trancamiento")
    elif short_count > 0:
        obs_parts.append("Parada por latas cortas")
    else:
        obs_parts.append("Parada por trancamiento trimmer")

    if resets:
        obs_parts.append(f"(Resets: {', '.join(resets)})")

    obs_text = " ".join(obs_parts)

    event_ext_id = (
        f"report_{machine_code}_{date_str}_{shift_code}_entry_{entry_slot}"
    )

    # 9. Create Event object
    report_event = EventWrite(
        external_id=event_ext_id,
        data_set_id=DATA_SET_ID,
        type="Production Report",
        subtype="Hourly Entry Standum",
        start_time=start_ms,
        end_time=end_ms,
        description=f"Production Report {hour_interval} for Standum {machine_code.upper()}",
        asset_ids=[asset_id],
        metadata={
            "timezone": "GMT-4",
            "machine_code": machine_code.upper(),
            "shift": shift_code,
            "hour_interval": hour_interval,
            "hourly_production": str(int(hourly_production)),
            "short_cans_per_hour": str(int(short_count)),
            "trimmer_jams_per_hour": str(int(trim_count)),
            "cans_by_short_can": str(int(cans_from_short)),
            "cans_by_trimmer_jam": str(int(cans_from_trim)),
            "downtime_by_short_can_min": f"{downtime_short:.2f}",
            "downtime_by_trimmer_jam_min": f"{downtime_trim:.2f}",
            "total_downtime_min": f"{total_downtime_min:.2f}",
            "merma_kg": f"{merma_kg:.2f}",
            "pct_merma": f"{pct_merma:.2f}%",
            "pct_eficiencia": f"{pct_eficiencia:.2f}%",
            "observations": obs_text,
        },
    )

    # 10. Upsert event to CDF
    res = client.events.upsert(report_event)
    ext_id = res.external_id if hasattr(res, "external_id") else res[0].external_id
    cdf_id = res.id if hasattr(res, "id") else res[0].id
    print(
        f"  --> Successfully posted Standum Event: '{ext_id}' (CDF ID: {cdf_id})"
    )

def generate_minster_event(
    client: CogniteClient,
    cfg: dict,
    start_ms: int,
    end_ms: int,
    last_hour_start_local: datetime,
    asset_id: int,
):
    """
    Dedicated generator for MINSTER Presses.
    Tracks both Coil Strokes (Golpes Bobina) and Shift Strokes (Golpes Turno).
    Calculates efficiency based on hourly coil strokes without scrap/mermas.
    """
    machine_code = cfg["code"]
    nominal_cap = cfg.get("nominal_capacity", 120000.0)

    start_hour_local = last_hour_start_local.hour

    # 1. Determine shift and start hour (Day: 06:00-18:00 | Night: 18:00-06:00)
    shift_code = "day" if 6 <= start_hour_local < 18 else "night"
    shift_start_hour = 6 if shift_code == "day" else 18

    # 2. Adjust operational shift date for overnight hours (00:00 - 05:59)
    if start_hour_local < 6:
        shift_date = last_hour_start_local - timedelta(days=1)
    else:
        shift_date = last_hour_start_local

    date_str = shift_date.strftime("%Y%m%d")

    # 3. Compute entry slot index (01 to 12)
    hour_index = ((start_hour_local - shift_start_hour) % 24) + 1
    entry_slot = f"{hour_index:02d}"

    # 4. Get human-readable hour label from map
    hour_interval = HOUR_INTERVAL_MAP.get(
        start_hour_local,
        f"{start_hour_local} a {(start_hour_local + 1) % 24}",
    )

    # 5. Query CDF for both production counters
    golpes_bob, bob_reset = calculate_hourly_counter_delta(
        client, cfg["ts_golpes_bob"], start_ms, end_ms
    )
    golpes_turno, turno_reset = calculate_hourly_counter_delta(
        client, cfg["ts_golpes_turno"], start_ms, end_ms
    )

    # Primary hourly production based on coil strokes
    hourly_production = golpes_bob

    # 6. Compute KPI metrics
    efficiency = (
        round((hourly_production * 100.0) / nominal_cap, 2)
        if nominal_cap > 0
        else 0.0
    )
    downtime_minutes = (
        max(0.0, round(60.0 - ((hourly_production * 60.0) / nominal_cap), 2))
        if nominal_cap > 0
        else 0.0
    )

    # 7. Resets tracking
    resets = []
    if bob_reset:
        resets.append("golpes_bobina")
    if turno_reset:
        resets.append("golpes_turno")

    obs_text = (
        f"Resets detectados: {', '.join(resets)}"
        if resets
        else "Operación normal"
    )

    event_ext_id = (
        f"report_{machine_code}_{date_str}_{shift_code}_entry_{entry_slot}"
    )

    # 8. Create Event object
    report_event = EventWrite(
        external_id=event_ext_id,
        data_set_id=DATA_SET_ID,
        type="Production Report",
        subtype="Hourly Entry MINSTER",
        start_time=start_ms,
        end_time=end_ms,
        description=f"Production Report {hour_interval} for MINSTER {machine_code.upper()}",
        asset_ids=[asset_id],
        metadata={
            "timezone": "GMT-4",
            "machine_code": machine_code.upper(),
            "shift": shift_code,
            "hour_interval": hour_interval,
            "golpes_bobina_hora": str(int(golpes_bob)),
            "golpes_turno_hora": str(int(golpes_turno)),
            "hourly_production": str(int(hourly_production)),
            "pct_eficiencia": f"{efficiency:.2f}%",
            "downtime_minutes": f"{downtime_minutes:.2f}",
            "observations": obs_text,
        },
    )

    # 9. Upsert event to CDF
    res = client.events.upsert(report_event)
    ext_id = res.external_id if hasattr(res, "external_id") else res[0].external_id
    cdf_id = res.id if hasattr(res, "id") else res[0].id
    print(
        f"  --> Successfully posted MINSTER Event: '{ext_id}' (CDF ID: {cdf_id})"
    )

def generate_ispray_event(
    client: CogniteClient,
    cfg: dict,
    start_ms: int,
    end_ms: int,
    last_hour_start_local: datetime,
    asset_id: int,
):
    """
    Dedicated generator for ISPRAY Machines.
    Calculates hourly production and efficiency percentage (No Scrap / Mermas).
    """
    machine_code = cfg["code"]
    nominal_cap = cfg.get("nominal_capacity", 30000.0)

    start_hour_local = last_hour_start_local.hour

    # 1. Determine shift and start hour (Day: 06:00-18:00 | Night: 18:00-06:00)
    shift_code = "day" if 6 <= start_hour_local < 18 else "night"
    shift_start_hour = 6 if shift_code == "day" else 18

    # 2. Adjust operational shift date for overnight hours (00:00 - 05:59)
    if start_hour_local < 6:
        shift_date = last_hour_start_local - timedelta(days=1)
    else:
        shift_date = last_hour_start_local

    date_str = shift_date.strftime("%Y%m%d")

    # 3. Compute entry slot index (01 to 12)
    hour_index = ((start_hour_local - shift_start_hour) % 24) + 1
    entry_slot = f"{hour_index:02d}"

    # 4. Get human-readable hour label from map
    hour_interval = HOUR_INTERVAL_MAP.get(
        start_hour_local,
        f"{start_hour_local} a {(start_hour_local + 1) % 24}",
    )

    # 5. Query CDF production counter
    hourly_production, prod_reset = calculate_hourly_counter_delta(
        client, cfg["ts_prod"], start_ms, end_ms
    )

    # 6. Compute KPI metrics (Production & Efficiency only)
    efficiency = (
        round((hourly_production * 100.0) / nominal_cap, 2)
        if nominal_cap > 0
        else 0.0
    )
    downtime_minutes = (
        max(0.0, round(60.0 - ((hourly_production * 60.0) / nominal_cap), 2))
        if nominal_cap > 0
        else 0.0
    )

    # 7. Resets tracking & Observations
    obs_text = "Reset detectado en contador" if prod_reset else "Operación normal"
    event_ext_id = (
        f"report_{machine_code}_{date_str}_{shift_code}_entry_{entry_slot}"
    )

    # 8. Create Event object targeting specific dataset
    report_event = EventWrite(
        external_id=event_ext_id,
        data_set_id=DATA_SET_ID,
        type="Production Report",
        subtype="Hourly Entry ISPRAY",
        start_time=start_ms,
        end_time=end_ms,
        description=f"Production Report {hour_interval} for ISPRAY {machine_code.upper()}",
        asset_ids=[asset_id],
        metadata={
            "timezone": "GMT-4",
            "machine_code": machine_code.upper(),
            "shift": shift_code,
            "hour_interval": hour_interval,
            "hourly_production": str(int(hourly_production)),
            "pct_eficiencia": f"{efficiency:.2f}%",
            "downtime_minutes": f"{downtime_minutes:.2f}",
            "observations": obs_text,
        },
    )

    # 9. Upsert event to CDF
    res = client.events.upsert(report_event)
    ext_id = res.external_id if hasattr(res, "external_id") else res[0].external_id
    cdf_id = res.id if hasattr(res, "id") else res[0].id
    print(
        f"  --> Successfully posted ISPRAY Event: '{ext_id}' (CDF ID: {cdf_id})"
    )


In [12]:
def run_all_production_reports(client: CogniteClient = None):
    # Fall back to global client instance if not explicitly provided
    if client is None:
        client = globals().get("client")
        if client is None:
            raise ValueError("CogniteClient instance 'client' was not found.")

    now_local = datetime.now(LOCAL_TZ)
    last_hour_end_local = now_local.replace(minute=0, second=0, microsecond=0)
    last_hour_start_local = last_hour_end_local - timedelta(hours=1)

    start_ms = int(last_hour_start_local.timestamp() * 1000)
    end_ms = int(last_hour_end_local.timestamp() * 1000)

    print("=" * 80)
    print(
        f"EXECUTION WINDOW (GMT-4): {last_hour_start_local.strftime('%Y-%m-%d %H:%M')} to {last_hour_end_local.strftime('%H:%M')}"
    )
    print("=" * 80)

    for cfg in MACHINE_CONFIGS:
        m_code = cfg["code"].upper()
        m_type_raw = cfg.get("machine_type", "")
        m_type_clean = m_type_raw.lower().strip()

        print(
            f"\n--- Processing [{m_type_raw.upper()}]: {m_code} ({cfg['asset_ext_id']}) ---"
        )

        try:
            asset_id = get_asset_id(client, cfg["asset_ext_id"])

            if m_type_clean == "printer":
                generate_printer_event(
                    client, cfg, start_ms, end_ms, last_hour_start_local, asset_id
                )
            elif m_type_clean == "standum":
                generate_standum_event(
                    client, cfg, start_ms, end_ms, last_hour_start_local, asset_id
                )
            elif m_type_clean == "di":
                generate_di_event(
                    client, cfg, start_ms, end_ms, last_hour_start_local, asset_id
                )
            elif m_type_clean == "minster":
                generate_minster_event(
                    client, cfg, start_ms, end_ms, last_hour_start_local, asset_id
                )
            elif m_type_clean == "ispray":
                generate_ispray_event(
                    client, cfg, start_ms, end_ms, last_hour_start_local, asset_id
                )
            else:
                print(f"  [Warning] Unsupported machine type: '{m_type_raw}'")

        except ValueError as err:
            print(f"  [Error] {err} Skipping {m_code}...")
            continue
        except Exception as err:
            print(f"  [Error] Unexpected exception for {m_code}: {err}. Skipping...")
            continue

run_all_production_reports(client)

EXECUTION WINDOW (GMT-4): 2026-08-21 18:00 to 19:00

--- Processing [PRINTER]: P11 (SuperenvasesMQTT_L1_PRINTER) ---
  [PRINTER_L1_PROD_ACT_DISP] First: 384,136.0 | Last: 37,190.0 | Delta: 37,190.0 | Reset: True
  [PRINTER_L1_RETRACT_ACT_DISP] First: 458.0 | Last: 68.0 | Delta: 68.0 | Reset: True
  [PRINTER_L1_LAT_SOP_ACT_DISP] First: 11,226.0 | Last: 966.0 | Delta: 966.0 | Reset: True
  --> Successfully posted Printer Event: 'report_p11_20260821_night_entry_01' (CDF ID: 8905071765237359)

--- Processing [PRINTER]: P31 (SuperenvasesMQTT_L3_PRINTER_PRINTER31) ---
  [Warning] [PRINTER_L31_PROD_ACT_DISP] No datapoints found in window.
  [Warning] [PRINTER_L31_RETRACT_ACT_DISP] No datapoints found in window.
  [Warning] [PRINTER_L31_LAT_SOP_ACT_DISP] No datapoints found in window.
  --> Successfully posted Printer Event: 'report_p31_20260821_night_entry_01' (CDF ID: 8651417353385216)

--- Processing [PRINTER]: P32 (SuperenvasesMQTT_L3_PRINTER_PRINTER32) ---
  [Warning] [PRINTER_L32_PROD_AC